# Family-level AME bubble plot + foreground overlap

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['svg.fonttype'] = 'none'

## Config

In [ ]:
# Resolve this analysis folder (paper/03_tfbs) regardless of the kernel's cwd.
def _here():
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / '00_build_ccre_universe.py').exists():
            return c
    raise RuntimeError('run this notebook from inside paper/03_tfbs')

DIR = _here()
MANIFEST = DIR / 'data' / 'pwms' / 'family_manifest.tsv'   # <- 02_build_family_bundle.py
AME      = DIR / 'results' / 'ame_family_long.parquet'     # <- 05_run_ame.py
RNA_FEAT = DIR / 'data' / 'gene_features.parquet'          # <- 03_prepare_rna.py
REGIONS  = DIR / 'data' / 'regions'                        # <- 01_prepare_regions.py
VST      = DIR.parent / '01_rna' / 'results' / 'gene_vst.parquet'

FIG_DIR = DIR / 'figs'
FIG_DIR.mkdir(parents=True, exist_ok=True)
FIG_NAME = 'ame_family_bubble'

for _p in (MANIFEST, AME, RNA_FEAT, VST):
    if not _p.exists():
        raise SystemExit(f'missing {_p} - run steps 00-05 (and 01_rna) first')

In [ ]:
STAGES = ['ESC', 'DE', 'HB', 'iHEP', 'mHEP']
CONTRASTS = ['DE_vs_ESC', 'HB_vs_DE', 'iHEP_vs_HB', 'mHEP_vs_iHEP']
DIRECTIONS = ['down', 'up']
CELL_LABELS = [f'{c}_{d}' for c in CONTRASTS for d in DIRECTIONS]

# Which stage is 'where the chromatin is accessible' for each cell
ACTIVE_STAGE = {
    ('DE_vs_ESC',  'up'): 'DE',  ('DE_vs_ESC',  'down'): 'ESC',
    ('HB_vs_DE',   'up'): 'HB',  ('HB_vs_DE',   'down'): 'DE',
    ('iHEP_vs_HB', 'up'): 'iHEP',('iHEP_vs_HB', 'down'): 'HB',
    ('mHEP_vs_iHEP','up'):'mHEP',('mHEP_vs_iHEP','down'): 'iHEP',
}

In [ ]:
# Filters
SIG_ALPHA          = 0.05    # AME adj_p cutoff
LOG2OR_MIN         = 0.5     # effect-size floor (motif enrichment)
TPM_AT_STAGE_MIN   = 1.0     # TF must be expressed at active stage
N_PER_FAMILY       = 3       # max surviving family members per cell

# Layout
CELL_W           = 1.4
CELL_H           = 0.90
SUBPOS_X         = [-0.30, 0.0, 0.30]    # x-offsets for rank 1/2/3 within a cell
LABEL_Y_OFFSET   = 0.32

# Bubble SIZE encodes TF VST at active stage
SIZE_VST_MIN     = 5.0
SIZE_VST_MAX     = 14.0
SIZE_MIN_PT2     = 40.0
SIZE_MAX_PT2     = 500.0

# Bubble COLOR encodes TF expression LFC at the contrast (single divergent map)
COLOR_LFC_VMIN   = -6.0
COLOR_LFC_VMAX   = 6.0
COLOR_CMAP       = 'RdBu_r'

# STROKE THICKNESS encodes log2 OR (motif effect size in chromatin)
STROKE_LOG2OR_MIN = 0.5
STROKE_LOG2OR_MAX = 3.0
STROKE_MIN_LW     = 0.4
STROKE_MAX_LW     = 2.5

## Load data + lookups

In [ ]:
manifest = pl.read_csv(MANIFEST, separator='\t')
ame = pl.read_parquet(AME)
fn = pl.col('pos') - pl.col('TP')
tn = pl.col('neg') - pl.col('FP')
ame = ame.with_columns(
    (((pl.col('TP') + 0.5) / (fn + 0.5)).log(base=2)
     - ((pl.col('FP') + 0.5) / (tn + 0.5)).log(base=2)).alias('log2_or'),
    (-pl.col('adj_p-value').log10()).alias('neg_log10_adj_p'),
)
print(f'manifest: {manifest.height} (family, TF) pairs across {manifest["family"].n_unique()} families')
print(f'ame: {ame.height} rows ({ame["motif_alt_ID"].n_unique()} motifs × {len(CELL_LABELS)} cells)')

In [ ]:
rna = pl.read_parquet(RNA_FEAT).select(
    ['gene_name'] + [f'tpm_{s}' for s in STAGES] + [f'lfc_{c}' for c in CONTRASTS]
)
tpm = {r['gene_name']: {s: float(r[f'tpm_{s}']) for s in STAGES}
       for r in rna.iter_rows(named=True)}
lfc = {r['gene_name']: {c: float(r[f'lfc_{c}']) for c in CONTRASTS}
       for r in rna.iter_rows(named=True)}

vst_raw = pl.read_parquet(VST)
vst_per_stage = vst_raw.with_columns([
    pl.mean_horizontal([c for c in vst_raw.columns if c.startswith(f'{s}_REP')]).alias(f'vst_{s}')
    for s in STAGES
]).select(['gene_name'] + [f'vst_{s}' for s in STAGES])
vst = {r['gene_name']: {s: float(r[f'vst_{s}']) for s in STAGES}
       for r in vst_per_stage.iter_rows(named=True)}

def _dimer(stage, table, agg=min):
    a = table.get('POU5F1', {}).get(stage)
    b = table.get('SOX2', {}).get(stage)
    return None if a is None or b is None else agg(a, b)

def get_tpm(tf, stage):
    return _dimer(stage, tpm) if tf == 'POU5F1::SOX2' else tpm.get(tf, {}).get(stage)

def get_vst(tf, stage):
    return _dimer(stage, vst) if tf == 'POU5F1::SOX2' else vst.get(tf, {}).get(stage)

def get_lfc(tf, contrast):
    if tf == 'POU5F1::SOX2':
        a = lfc.get('POU5F1', {}).get(contrast)
        b = lfc.get('SOX2', {}).get(contrast)
        return None if a is None or b is None else 0.5 * (a + b)
    return lfc.get(tf, {}).get(contrast)

## Apply filters + rank top-3 per (family, cell) by VST

In [ ]:
joined = manifest.join(ame, left_on='alt_id', right_on='motif_alt_ID', how='left')

# Annotate each row with active-stage stats + per-contrast LFC
rows = []
for r in joined.iter_rows(named=True):
    stage = ACTIVE_STAGE.get((r['contrast'], r['direction']))
    atpm = get_tpm(r['tf'], stage) if stage else None
    avst = get_vst(r['tf'], stage) if stage else None
    clfc = get_lfc(r['tf'], r['contrast'])
    rows.append({**r,
                 'active_stage': stage,
                 'active_stage_tpm': atpm,
                 'active_stage_vst': avst,
                 'contrast_lfc': clfc,
                 'expressed_at_active': atpm is not None and atpm >= TPM_AT_STAGE_MIN})
joined = pl.DataFrame(rows)

# Apply filters
survivors = joined.filter(
    pl.col('adj_p-value') < SIG_ALPHA,
    pl.col('log2_or') >= LOG2OR_MIN,
    pl.col('expressed_at_active'),
    pl.col('active_stage_vst').is_not_null(),
)

# Top-N per (family, cell) by active-stage VST
ranked = (
    survivors
    .sort('active_stage_vst', descending=True, nulls_last=True)
    .with_columns(
        pl.int_range(1, pl.len() + 1).over(['family', 'contrast', 'direction']).alias('vst_rank')
    )
    .filter(pl.col('vst_rank') <= N_PER_FAMILY)
)
print(f'Surviving rows after filters: {ranked.height}  (across {ranked.select(["family","contrast","direction"]).unique().height} family×cell positions)')

## Plot

In [ ]:
def size_from_vst(v):
    if v is None or np.isnan(v):
        return SIZE_MIN_PT2
    v = np.clip(v, SIZE_VST_MIN, SIZE_VST_MAX)
    return SIZE_MIN_PT2 + (v - SIZE_VST_MIN) / (SIZE_VST_MAX - SIZE_VST_MIN) * (SIZE_MAX_PT2 - SIZE_MIN_PT2)

def stroke_from_log2or(v):
    v = np.clip(v, STROKE_LOG2OR_MIN, STROKE_LOG2OR_MAX)
    return STROKE_MIN_LW + (v - STROKE_LOG2OR_MIN) / (STROKE_LOG2OR_MAX - STROKE_LOG2OR_MIN) * (STROKE_MAX_LW - STROKE_MIN_LW)

In [ ]:
fam_order = manifest['family'].unique(maintain_order=True).to_list()
by_cell = {(r['family'], r['contrast'], r['direction'], r['vst_rank']): r
           for r in ranked.iter_rows(named=True)}

n_fams = len(fam_order)
n_cells = len(CELL_LABELS)
fig_w = CELL_W * n_cells + 5.0
fig_h = CELL_H * n_fams + 1.8
fig, ax = plt.subplots(figsize=(fig_w, fig_h))

ax.set_xlim(-0.5, n_cells - 0.5)
ax.set_ylim(n_fams - 0.5, -0.5)
ax.set_aspect('auto')
ax.set_xticks(np.arange(-0.5, n_cells, 1), minor=True)
ax.set_yticks(np.arange(-0.5, n_fams, 1), minor=True)
ax.grid(which='minor', color='#dddddd', linewidth=0.5)
ax.tick_params(which='minor', length=0)

col_dirs = np.array([label.rsplit('_', 1)[1] for label in CELL_LABELS])
cmap = plt.get_cmap(COLOR_CMAP)
norm = plt.matplotlib.colors.Normalize(vmin=COLOR_LFC_VMIN, vmax=COLOR_LFC_VMAX)
fam_to_row = {f: i for i, f in enumerate(fam_order)}

for label_idx, label in enumerate(CELL_LABELS):
    contrast, direction = label.rsplit('_', 1)
    for fam in fam_order:
        fam_y = fam_to_row[fam]
        for rank_pos in range(1, N_PER_FAMILY + 1):
            rec = by_cell.get((fam, contrast, direction, rank_pos))
            if rec is None:
                continue
            x = label_idx + SUBPOS_X[rank_pos - 1]
            size = size_from_vst(rec.get('active_stage_vst'))
            lfc_val = rec.get('contrast_lfc')
            if lfc_val is None:
                color = (0.85, 0.85, 0.85, 1.0)
            else:
                color = cmap(norm(np.clip(lfc_val, COLOR_LFC_VMIN, COLOR_LFC_VMAX)))
            stroke = stroke_from_log2or(rec['log2_or'])
            ax.scatter(x, fam_y, s=size, c=[color],
                       edgecolors='black', linewidths=stroke, zorder=3)
            ax.text(x, fam_y - LABEL_Y_OFFSET, rec['tf'],
                    ha='center', va='bottom',
                    fontsize=7, color='black', fontweight='bold', zorder=4)

# X axis
ax.set_xticks(range(n_cells))
ax.set_xticklabels(col_dirs, fontsize=9)
ax.tick_params(axis='x', length=0, pad=4, which='major')

# Y axis
ax.set_yticks(range(n_fams))
ax.set_yticklabels(fam_order, fontsize=9, family='monospace')
ax.tick_params(axis='y', length=0, pad=4, which='major')
for sp in ax.spines.values():
    sp.set_visible(False)

# Top transition titles
for k, contrast in enumerate(CONTRASTS):
    a, b = contrast.split('_vs_')
    ax.text(2 * k + 0.5, -1.0, f'{b} → {a}', ha='center', va='bottom',
            fontsize=10, fontweight='bold')
    ax.plot([2 * k - 0.4, 2 * k + 1.4], [-0.65, -0.65],
            color='black', lw=0.8, clip_on=False)
for k in range(2, n_cells, 2):
    ax.axvline(k - 0.5, color='black', lw=0.5, alpha=0.5)

# Color legend (LFC)
cax = ax.inset_axes([1.03, 0.30, 0.020, 0.40])
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
cb = fig.colorbar(sm, cax=cax)
cb.set_label('TF expression log2FC at this contrast', fontsize=8)
cb.ax.tick_params(labelsize=7)
cb.ax.axhline(0, color='black', lw=0.4)

# Size legend (VST)
leg_ax = ax.inset_axes([1.02, 0.92, 0.16, 0.06])
leg_ax.set_xlim(0, 4); leg_ax.set_ylim(0, 1); leg_ax.axis('off')
for i, v in enumerate([6.0, 9.0, 12.0, 14.0]):
    leg_ax.scatter(i + 0.5, 0.55, s=size_from_vst(v), c='lightgray',
                   edgecolors='black', linewidths=0.4)
    leg_ax.text(i + 0.5, -0.05, f'{v:g}', ha='center', va='bottom', fontsize=6)
leg_ax.text(2.0, 1.05, 'VST at active stage (bubble size)',
            ha='center', va='bottom', fontsize=8)

# Stroke legend (log2 OR)
stroke_leg = ax.inset_axes([1.02, 0.84, 0.16, 0.06])
stroke_leg.set_xlim(0, 4); stroke_leg.set_ylim(0, 1); stroke_leg.axis('off')
fixed_size = (SIZE_MIN_PT2 + SIZE_MAX_PT2) / 2
for i, v in enumerate([0.5, 1.0, 2.0, 3.0]):
    stroke_leg.scatter(i + 0.5, 0.55, s=fixed_size, c='lightgray',
                       edgecolors='black', linewidths=stroke_from_log2or(v))
    stroke_leg.text(i + 0.5, -0.05, f'{v:g}', ha='center', va='bottom', fontsize=6)
stroke_leg.text(2.0, 1.05, 'log2 OR (stroke thickness)',
                ha='center', va='bottom', fontsize=8)

fig.text(0.99, 0.005,
         f'Per (family × cell): up to {N_PER_FAMILY} surviving family members '
         f'(adj_p < {SIG_ALPHA}, log2OR ≥ {LOG2OR_MIN}, TPM at active stage ≥ {TPM_AT_STAGE_MIN}), '
         f'left-to-right by descending VST. '
         f'Size = TF VST; color = TF RNA log2FC at contrast; stroke = log2 OR.',
         ha='right', va='bottom', fontsize=7, style='italic')


fig.savefig(FIG_DIR / f'{FIG_NAME}.png', dpi=200, bbox_inches='tight')
fig.savefig(FIG_DIR / f'{FIG_NAME}.pdf', bbox_inches='tight')
print('->', FIG_DIR / f'{FIG_NAME}.pdf')

## Foreground overlap - do consecutive transitions act on the same cCREs?

In [ ]:
TRANSITIONS = [('ESC→DE', 'DE_vs_ESC'), ('DE→HB', 'HB_vs_DE'),
               ('HB→iHLC', 'iHEP_vs_HB'), ('iHLC→HLC', 'mHEP_vs_iHEP')]


def load_fg(stem):
    p = REGIONS / f'fg_{stem}.bed'
    if not p.exists():
        raise SystemExit(f'missing {p} - run 01_prepare_regions.py first')
    return {tuple(l.split('\t')[:3]) for l in p.read_text().splitlines() if l.strip()}


sets = {f'{lab} {d}': load_fg(f'{stem}_{d}')
        for lab, stem in TRANSITIONS for d in ('up', 'down')}
for k, v in sets.items():
    print(f'  {k:<18} {len(v):>7,} regions')

In [ ]:
from matplotlib_venn import venn2, venn2_circles

C1, C2 = '#2C6FB5', '#EE5A24'
ups = [f'{lab} up' for lab, _ in TRANSITIONS]
downs = [u.replace(' up', ' down') for u in ups]


def venn_panel(ax, a, b):
    A, B = sets[a], sets[b]
    sfx = ' up' if a.endswith(' up') else ' down'
    na, nb = a.replace(sfx, ''), b.replace(sfx, '')
    only_a, only_b, both = len(A - B), len(B - A), len(A & B)
    v = venn2(subsets=(only_a, only_b, both), ax=ax,
              set_labels=(na, nb), set_colors=(C1, C2), alpha=0.55)
    venn2_circles(subsets=(only_a, only_b, both), ax=ax, lw=0.8, color='0.25')
    for lbl, txt in zip(('10', '01', '11'),
                        (f'{only_a:,}', f'{only_b:,}', f'{both:,}')):
        t = v.get_label_by_id(lbl)
        if t is not None:
            t.set_text(txt); t.set_fontsize(8)
    for t in v.set_labels or []:
        if t is not None:
            t.set_fontsize(9)
    ax.set_title(f'shared {both:,}  ·  J = {both / len(A | B):.3f}\n'
                 f'{both / len(A):.1%} of {na}, {both / len(B):.1%} of {nb}',
                 fontsize=8)


rows = [('opening (up) sets', list(zip(ups, ups[1:]))),
        ('closing (down) sets', list(zip(downs, downs[1:])))]
fig, axes = plt.subplots(2, 3, figsize=(12.6, 8.2))
for r, (rlab, pairs) in enumerate(rows):
    for ax, (a, b) in zip(axes[r], pairs):
        venn_panel(ax, a, b)
    axes[r][0].text(-0.14, 0.5, rlab, transform=axes[r][0].transAxes,
                    rotation=90, va='center', ha='center',
                    fontsize=9.5, fontweight='bold')
fig.suptitle('cCRE sets at consecutive transitions — circle area ∝ number of elements',
             fontsize=10.5, x=0.02, ha='left')
fig.tight_layout(rect=(0.02, 0, 1, 0.95))

fig.savefig(FIG_DIR / 'fg_overlap_venn.png', dpi=200)
fig.savefig(FIG_DIR / 'fg_overlap_venn.pdf')
print('->', FIG_DIR / 'fg_overlap_venn.pdf')
fig